Now that I have pushed some collections, I can use them in code to do my 
analysis, fully delegating I/O management to hippo! Most importantly, it does not
matter if I am on a different machine: hippo gaurantees portability and reproducibility.

In [ ]:
from henry import Henry
from henry.collection import CollectionInstance
import hippometa as hm 

from pixell import enmap, enplot, curvedsky
import numpy as np
from matplotlib import pyplot as plt

client = Henry()

I will demo a simple approximate power spectrum for pa6 f150. Let's grab the top-level collection (we will dynamically filter it for individual products):
* Still need to know its id
* I do `realize_sources=True`. After doing this once (or manually caching the collection in advance), hippo will know not to re-download products!

In [ ]:
mycol = client.pull_collection('6ad18e65b4c52fc5e10d49c8', realize_sources=True, pull_children=True)

Ignoring my mess-up with the metadata of these windows (they only look different
than the maps in their metdata because splits is `None`, this is quite fragile),
I can now write a script that interacts solely with this dictionary!

This is very basic power spectra, that is besides the point...

In [ ]:
lmax = 2000

dr6_pa6_f150_kwargs = dict(
    instrument='pa6',
    frequency='f150'
)

window_prod = mycol.filter(metadata_type=hm.MapSet, metadata_filter={'split': None, **dr6_pa6_f150_kwargs}).one()
beam_prod = mycol.filter(metadata_type=hm.BeamMetadata, metadata_filter=dr6_pa6_f150_kwargs).one()

w = enmap.read_map(str(window_prod['map'].path))
l, bl = np.loadtxt(beam_prod['data'].path, usecols=(0, 1)).T

l = l[:lmax+1]
bl = bl[:lmax + 1]
bl /= bl[0]

# get the 4 splits! sel=0 just loads the intensity component for simplicity
imap_prods = [mycol.filter(metadata_type=hm.MapSet, metadata_filter={'split': i, **dr6_pa6_f150_kwargs}).one() for i in range(4)]
imaps = [enmap.read_map(str(imap_prod['source_free'].path), sel=0) for imap_prod in imap_prods]

# get my "pseudospectrum to powerspectrum" operator
w2 = np.sum(w**2 * w.pixsizemap()) / 4 / np.pi 
fac = l * (l + 1) / 2 / np.pi / bl**2 / w2

# get my alms
alms = [curvedsky.map2alm(w * imaps[i], lmax=lmax) for i in range(4)]

# get a cl from the average of split crosses
cl = 0 
count = 0
for i in range(4):
    for j in range(i + 1, 4): # no autos!
        cl += curvedsky.alm2cl(alms[i], alms[j])
        count += 1
cl /= count
cl *= fac

# what do we got
plt.plot(l, cl)
plt.ylabel('Dl_TT')
plt.title('pa6_f150 power spectrum')